# Day 02 Demo: Prompt Engineering & Structured Outputs

In this lab, we master:
1. System and user message framing.
2. Controlling generation temperature and token limits.
3. Enforcing JSON schemas with Pydantic validation.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath('..'))

from pydantic import BaseModel, Field
from typing import List, Optional
from shared.utils import UnifiedLLMClient, count_tokens

client = UnifiedLLMClient()
print("LLM Client ready:", client.provider, client.model)

## 1. Defining Target Pydantic Schema

In [ ]:
class ExtractedFeedback(BaseModel):
    product: str = Field(description="Product or service name")
    sentiment: str = Field(description="Positive, Negative, or Mixed")
    key_issues: List[str] = Field(default_factory=list)
    refund_requested: bool = Field(default=False)

schema_str = json.dumps(ExtractedFeedback.model_json_schema(), indent=2)
print("Target JSON Schema:\n", schema_str)

## 2. Ingesting Unstructured Text & Extracting Validated JSON

In [ ]:
raw_input = (
    "I bought the Apex Cloud Router on Oct 15. The 5GHz signal drops every 20 minutes, "
    "and the unit runs dangerously hot to the touch. I demand a full refund immediately!"
)

system_prompt = (
    "You are a precise data extraction tool. Extract information from the user feedback "
    "strictly according to this JSON Schema:\n" + schema_str + "\nOutput ONLY valid JSON."
)

raw_output = client.complete(raw_input, system_prompt=system_prompt, json_mode=True, temperature=0.1)
print("Raw Model Output:", raw_output)

# Validate using Pydantic
try:
    parsed_data = json.loads(raw_output)
    validated_obj = ExtractedFeedback.model_validate(parsed_data)
    print("\n--- Successfully Parsed & Validated Pydantic Object ---")
    print("Product:", validated_obj.product)
    print("Sentiment:", validated_obj.sentiment)
    print("Refund Requested:", validated_obj.refund_requested)
    print("Key Issues:", validated_obj.key_issues)
except Exception as e:
    print("Schema validation error:", e)